# WEEK 4: PREDICTIVE MODELING FOR TOURISM DEMAND FORECASTING
### End-to-End Econometric Time Series, Regularized Regression, Tree Ensembles, and Multi-Scenario Policy Projections (2025–2027)

**Student**: Akshita | **Programme**: B.Tech Computer Science & Engineering  
**Institution**: Amritsar Group of Colleges  
**Project**: Yuva Internship - Tourism Data Science & Analytics (Week 4 Deliverable)  
**Data Source**: Official Ministry of Tourism & Bureau of Immigration (Tourism_raw_data.pdf)  
**Champion Architecture**: Blended Hybrid Ensemble (Ridge 50% + SARIMA 30% + Random Forest 20%)


In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Data ingestion
DATA_DIR = Path('../data')
df_annual = pd.read_csv(DATA_DIR / 'annual_series_2001_2024.csv')
df_quarterly = pd.read_csv(DATA_DIR / 'quarterly_series_2001_2024.csv')
df_monthly = pd.read_csv(DATA_DIR / 'monthly_demand_features_2019_2024.csv')

print(f'Ingested {len(df_annual)} annual records (2001-2024).')
print(f'Ingested {len(df_quarterly)} quarterly records (2011Q1-2024Q4).')
print(f'Ingested {len(df_monthly)} high-frequency monthly feature rows.')
df_quarterly.head(5)


## 1. Feature Engineering & Autoregressive Formulation
We construct an autoregressive and macroeconomic feature store:
- **Seasonal harmonics**: Q1, Q2, Q3, Q4 dummy indicators
- **Autoregressive lags**: Lag 1 (prior quarter), Lag 2, Lag 4 (year-over-year annual seasonal lag)
- **Macroeconomic exogenous regressors**: Implied USD/INR exchange rate, World tourism arrivals growth (%)
- **Structural shock indicators**: COVID shock indicator (2020-2021), peak lockdown dummy (2020Q2-Q3), post-pandemic recovery phase (2022+)


In [2]:
feature_cols = [
    'q1_dummy', 'q2_dummy', 'q3_dummy', 'q4_dummy',
    'covid_shock', 'covid_lockdown_peak', 'recovery_phase',
    'implied_usd_inr', 'world_arr_growth_pct',
    'fta_lag1', 'fta_lag2', 'fta_lag4'
]
for c in ['fta_lag1', 'fta_lag2', 'fta_lag4']:
    df_quarterly[c] = df_quarterly[c].bfill()

train_mask = df_quarterly['year'] <= 2022
test_mask = df_quarterly['year'] >= 2023

X_train = df_quarterly.loc[train_mask, feature_cols].values
y_train = df_quarterly.loc[train_mask, 'fta_volume'].values
X_test = df_quarterly.loc[test_mask, feature_cols].values
y_test = df_quarterly.loc[test_mask, 'fta_volume'].values

print(f'Training window: {len(y_train)} quarters (2011Q1 to 2022Q4)')
print(f'Out-of-sample holdout test window: {len(y_test)} quarters (2023Q1 to 2024Q4)')


## 2. Model Training & Benchmarking
We benchmark across 9 distinct model families under a strictly leak-free temporal split:
1. Baseline: Naive Seasonal Drift
2. Holt-Winters Triple Exponential Smoothing
3. Univariate SARIMA(1,0,1)x(0,1,1)_4
4. Econometric SARIMAX with Structural Shock Regressors
5. Regularized Ridge Regression (L2 Penalty)
6. ElasticNet Regression (L1 + L2 Penalty)
7. Random Forest Regressor
8. HistGradientBoosting Regressor
9. Champion Blended Hybrid Ensemble (Ridge 50% + SARIMA 30% + RF 20%)


In [3]:
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

# Fit Ridge
ridge = Ridge(alpha=10.0, random_state=42).fit(X_train_sc, y_train)
p_ridge = ridge.predict(X_test_sc)

# Fit SARIMA
sarima = SARIMAX(y_train, order=(1,0,1), seasonal_order=(0,1,1,4),
                 enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
p_sarima = sarima.forecast(steps=len(y_test))

# Fit Random Forest
rf = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42).fit(X_train, y_train)
p_rf = rf.predict(X_test)

# Fit Holt-Winters
hw = ExponentialSmoothing(y_train, seasonal_periods=4, trend='add', seasonal='add', damped_trend=True).fit(optimized=True)
p_hw = hw.forecast(len(y_test))

# Blended Champion
p_blend = 0.50 * p_ridge + 0.30 * p_sarima + 0.20 * p_rf

print('Model training and inference complete.')


## 3. Performance Metrics on Holdout (2023Q1–2024Q4)
Comparing MAE, RMSE, MAPE (%), R-squared, and Directional Accuracy (MDA %).

In [4]:
df_metrics = pd.read_csv('../metrics/model_evaluation_metrics.csv')
print('=== Out-of-Sample Holdout Performance (2023Q1 to 2024Q4) ===')
df_metrics


## 4. Residual Diagnostics
Testing statistical validity: Durbin-Watson for serial correlation, Jarque-Bera for residual normality, and Ljung-Box test for white noise residuals.

In [5]:
with open('../metrics/residual_diagnostics.json') as f:
    import json
    diag = json.load(f)
print(json.dumps(diag, indent=2))


## 5. Multi-Scenario Forward Forecasting (2025–2027)
Projecting demand under 3 macro scenarios:
- **Baseline**: Historical recovery momentum, +4.5% global growth
- **Optimistic**: Rapid e-Visa expansion, Incredible India 3.0, +6.5% global growth
- **Pessimistic**: Geopolitical friction in South Asia, fuel cost inflation, +2.0% global growth


In [6]:
df_scenarios = pd.read_csv('../data/forecast_results_2025_2027.csv')
annual_summary = df_scenarios.groupby(['scenario', 'year']).agg({
    'projected_fta': 'sum',
    'fee_usd_million': 'sum',
    'fee_inr_crore': 'sum'
}).reset_index()
print('=== Annual Forecast Summary (2025 - 2027) ===')
annual_summary
